# Memory Practice Notebook

Use this notebook to test memory subject IDs, access policies, and optimization metadata.

## 1. Setup

Before running:
1. Configure OCI credentials in `sandbox.yaml`.
2. Install dependencies (`uv sync`).
3. Keep the same `MEMORY_SUBJECT_ID` to test recall across conversations.

In [ ]:
import time
from openai import OpenAI
from openai_sdk.openai_client_provider import OpenAIClientProvider

client: OpenAI = OpenAIClientProvider().oci_openai_client
MODEL_ID = 'openai.gpt-4.1'
MEMORY_SUBJECT_ID = 'user_123456'
print('Client ready')

## 2. Memory Subject Recall (store + recall)

In [ ]:
conversation_a = client.conversations.create(metadata={'memory_subject_id': MEMORY_SUBJECT_ID})
response_a = client.responses.create(
    model=MODEL_ID,
    input="I like fish. I don't like shrimp.",
    conversation=conversation_a.id,
)
print('Conversation A:', conversation_a.id)
print('Model:', response_a.output_text)

In [ ]:
time.sleep(10)  # allow memory indexing

In [ ]:
conversation_b = client.conversations.create(metadata={'memory_subject_id': MEMORY_SUBJECT_ID})
response_b = client.responses.create(
    model=MODEL_ID,
    input='What do I like?',
    conversation=conversation_b.id,
)
print('Conversation B:', conversation_b.id)
print('Model:', response_b.output_text)

## 3. Access Policy Behavior (`store_only` then `recall_only`)

In [ ]:
store_only_conversation = client.conversations.create(
    metadata={
        'memory_subject_id': MEMORY_SUBJECT_ID,
        'memory_access_policy': 'store_only',
    }
)
client.responses.create(
    model=MODEL_ID,
    input='My favorite snack is mango slices.',
    conversation=store_only_conversation.id,
)
print('Stored message in store_only conversation')

In [ ]:
time.sleep(10)

In [ ]:
recall_only_conversation = client.conversations.create(
    metadata={
        'memory_subject_id': MEMORY_SUBJECT_ID,
        'memory_access_policy': 'recall_only',
    }
)
recall_response = client.responses.create(
    model=MODEL_ID,
    input='What snack do I like?',
    conversation=recall_only_conversation.id,
)
recall_response.output_text

## 4. Short-Term Memory Optimization Metadata

In [ ]:
optimized_conversation = client.conversations.create(
    metadata={
        'topic': 'demo',
        'short_term_memory_optimization': 'True',
    },
    items=[{'type': 'message', 'role': 'user', 'content': 'Hello!'}],
)
optimized_conversation